# Understanding the Ornstein-Uhlenbeck Process

A visual, hands-on introduction to mean-reverting stochastic processes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/ou-process-tutorial/blob/main/ou_process_tutorial.ipynb)

---

## What is the OU Process?

The Ornstein-Uhlenbeck process models a particle connected to equilibrium by a **"spring"** while being buffeted by **random noise**:

$$dX_t = \theta(\mu - X_t)dt + \sigma dW_t$$

| Parameter | Meaning | Intuition |
|-----------|---------|----------|
| $\theta$ | Mean-reversion speed | How fast the spring pulls back |
| $\mu$ | Long-term mean | Equilibrium position |
| $\sigma$ | Volatility | Noise intensity |

**Think of it as:** A drunk person on a rubber band — always pulled toward home but constantly stumbling.

## Setup

Run this cell first to import everything we need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, Optional

# Nice dark theme for plots
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Colors
TEAL = '#00d4aa'
CORAL = '#ff6b6b'
PURPLE = '#a855f7'
AMBER = '#fbbf24'

print('Setup complete!')

---

## Part 1: The Simulation

We simulate the OU process using the **Euler-Maruyama** method:

$$X_{t+dt} = X_t + \theta(\mu - X_t)dt + \sigma \sqrt{dt} \cdot Z$$

where $Z \sim N(0,1)$.

In [ ]:
def simulate_ou(
    theta: float,    # Mean-reversion speed
    mu: float,       # Long-term mean
    sigma: float,    # Volatility
    x0: float,       # Starting position
    T: float = 10.0, # Total time
    dt: float = 0.01,# Time step
    n_paths: int = 1,# Number of simulations
    seed: int = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Simulate 1D Ornstein-Uhlenbeck process.
    
    Returns:
        t: time points
        X: simulated paths (n_paths x n_steps)
    """
    if seed is not None:
        np.random.seed(seed)
    
    n_steps = int(T / dt) + 1
    t = np.linspace(0, T, n_steps)
    X = np.zeros((n_paths, n_steps))
    X[:, 0] = x0
    
    sqrt_dt = np.sqrt(dt)
    
    for i in range(1, n_steps):
        Z = np.random.randn(n_paths)  # Random noise
        drift = theta * (mu - X[:, i-1]) * dt
        diffusion = sigma * sqrt_dt * Z
        X[:, i] = X[:, i-1] + drift + diffusion
    
    return t, X

print('Simulation function ready!')

### Try it: Your First OU Simulation

In [ ]:
# Simulate one path
t, X = simulate_ou(theta=0.5, mu=0, sigma=0.3, x0=3, T=20, n_paths=1, seed=42)

plt.figure(figsize=(12, 5))
plt.plot(t, X[0], color=TEAL, linewidth=1.5)
plt.axhline(0, color=CORAL, linestyle='--', linewidth=2, label='μ = 0 (equilibrium)')
plt.xlabel('Time')
plt.ylabel('X(t)')
plt.title('Single OU Process Path', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---

## Part 2: Key Theoretical Results

The OU process has beautiful closed-form solutions:

| Property | Formula |
|----------|--------|
| Expected value | $E[X_t] = \mu + (x_0 - \mu)e^{-\theta t}$ |
| Variance | $\text{Var}(X_t) = \frac{\sigma^2}{2\theta}(1 - e^{-2\theta t})$ |
| Relaxation time | $\tau = 1/\theta$ |
| Stationary variance | $\sigma_\infty^2 = \sigma^2 / 2\theta$ |

In [ ]:
@dataclass
class OUProperties:
    """Theoretical properties of OU process."""
    theta: float
    mu: float
    sigma: float
    relaxation_time: float      # τ = 1/θ
    stationary_variance: float  # σ²/(2θ)
    stationary_std: float

def get_properties(theta, mu, sigma):
    """Compute theoretical OU properties."""
    tau = 1 / theta
    stat_var = sigma**2 / (2 * theta)
    return OUProperties(
        theta=theta, mu=mu, sigma=sigma,
        relaxation_time=tau,
        stationary_variance=stat_var,
        stationary_std=np.sqrt(stat_var)
    )

def expected_value(t, x0, mu, theta):
    """E[X_t] = μ + (x₀ - μ)e^(-θt)"""
    return mu + (x0 - mu) * np.exp(-theta * t)

def variance(t, theta, sigma):
    """Var(X_t) = (σ²/2θ)(1 - e^(-2θt))"""
    return (sigma**2 / (2 * theta)) * (1 - np.exp(-2 * theta * t))

# Example
props = get_properties(theta=0.5, mu=0, sigma=0.3)
print(f"θ = {props.theta}")
print(f"Relaxation time τ = {props.relaxation_time:.2f} seconds")
print(f"Stationary std σ_∞ = {props.stationary_std:.3f}")

---

## Part 3: Visualizing Mean Reversion

The key insight: **all paths get pulled back to μ**.

In [ ]:
# Parameters
theta, mu, sigma, x0 = 0.3, 0, 0.4, 4.0
T = 25

# Simulate many paths
t, X = simulate_ou(theta, mu, sigma, x0, T=T, n_paths=50, seed=42)
props = get_properties(theta, mu, sigma)

# Plot
fig, ax = plt.subplots(figsize=(14, 7))

# All paths (faded)
for i in range(50):
    ax.plot(t, X[i], color=PURPLE, alpha=0.3, linewidth=0.8)

# Highlight a few
for i in [0, 15, 30]:
    ax.plot(t, X[i], color=TEAL, alpha=0.9, linewidth=2)

# Theoretical mean
mean_theory = expected_value(t, x0, mu, theta)
ax.plot(t, mean_theory, 'w--', linewidth=3, label=r'$E[X_t] = \mu + (x_0-\mu)e^{-\theta t}$')

# Equilibrium line
ax.axhline(mu, color=CORAL, linewidth=2, label=f'μ = {mu} (equilibrium)')

# Confidence band
std_theory = np.sqrt(variance(t, theta, sigma))
ax.fill_between(t, mean_theory - 2*std_theory, mean_theory + 2*std_theory, 
                alpha=0.15, color=TEAL, label='±2σ band')

# Relaxation time marker
tau = props.relaxation_time
ax.axvline(tau, color=AMBER, linestyle=':', linewidth=2, alpha=0.7)
ax.annotate(f'τ = {tau:.1f}s', xy=(tau, 3), fontsize=12, color=AMBER)

ax.set_xlabel('Time (t)', fontsize=13)
ax.set_ylabel('X(t)', fontsize=13)
ax.set_title(f'Mean Reversion in OU Process\nθ={theta}, μ={mu}, σ={sigma}', 
             fontsize=15, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(alpha=0.3)
plt.show()

print(f"\n After time τ = {tau:.1f}s, the process has 'forgotten' ~63% of its initial displacement.")

---

## Part 4: Effect of θ (Mean-Reversion Speed)

**Higher θ = faster return to equilibrium**

 **Try changing the values below!**

In [ ]:
#  MODIFY THESE VALUES
theta_values = [0.1, 0.5, 2.0]  # Try: [0.05, 0.2, 1.0] or [0.3, 1.0, 5.0]

# Fixed parameters
mu, sigma, x0 = 0, 0.5, 5.0
T = 20

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [TEAL, PURPLE, AMBER]

for ax, theta, color in zip(axes, theta_values, colors):
    t, X = simulate_ou(theta, mu, sigma, x0, T=T, n_paths=30, seed=42)
    tau = 1 / theta
    
    # Plot paths
    for i in range(30):
        ax.plot(t, X[i], color=color, alpha=0.3, linewidth=0.8)
    
    # Mean
    ax.plot(t, expected_value(t, x0, mu, theta), 'w--', linewidth=2)
    ax.axhline(mu, color=CORAL, linewidth=1.5)
    ax.axvline(tau, color='white', linestyle=':', alpha=0.5)
    
    ax.set_xlabel('Time')
    ax.set_ylabel('X(t)' if ax == axes[0] else '')
    ax.set_title(f'θ = {theta}\nτ = {tau:.1f}s', fontsize=13, color=color)
    ax.set_ylim(-4, 7)
    ax.grid(alpha=0.3)

fig.suptitle('Effect of Mean-Reversion Speed (θ)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Part 5: Effect of σ (Volatility)

**Higher σ = wider fluctuations around μ**

The stationary standard deviation is $\sigma_\infty = \sigma / \sqrt{2\theta}$

In [ ]:
#  MODIFY THESE VALUES
sigma_values = [0.1, 0.5, 1.5]  # Try: [0.2, 0.8, 2.0]

# Fixed parameters
theta, mu, x0 = 0.3, 0, 3.0
T = 25

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [TEAL, PURPLE, AMBER]

for ax, sigma, color in zip(axes, sigma_values, colors):
    t, X = simulate_ou(theta, mu, sigma, x0, T=T, n_paths=30, seed=42)
    props = get_properties(theta, mu, sigma)
    stat_std = props.stationary_std
    
    # Plot paths
    for i in range(30):
        ax.plot(t, X[i], color=color, alpha=0.3, linewidth=0.8)
    
    # Equilibrium and bands
    ax.axhline(mu, color=CORAL, linewidth=2)
    ax.axhline(mu + 2*stat_std, color='white', linestyle='--', alpha=0.5)
    ax.axhline(mu - 2*stat_std, color='white', linestyle='--', alpha=0.5)
    ax.fill_between(t, mu - 2*stat_std, mu + 2*stat_std, alpha=0.1, color=color)
    
    ax.set_xlabel('Time')
    ax.set_ylabel('X(t)' if ax == axes[0] else '')
    ax.set_title(f'σ = {sigma}\nσ_∞ = {stat_std:.2f}', fontsize=13, color=color)
    ax.set_ylim(-7, 9)
    ax.grid(alpha=0.3)

fig.suptitle(f'Effect of Volatility (σ) with θ = {theta}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Part 6: Stationary Distribution

After a long time, the process **forgets its starting point** and settles into:

$$X_\infty \sim \mathcal{N}\left(\mu, \frac{\sigma^2}{2\theta}\right)$$

In [ ]:
# Parameters
theta, mu, sigma, x0 = 0.4, 2.0, 0.7, -3.0
T = 50

# Simulate many paths
t, X = simulate_ou(theta, mu, sigma, x0, T=T, n_paths=500, seed=42)
props = get_properties(theta, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})

# Left: trajectories
ax1 = axes[0]
for i in range(50):  # Show subset
    ax1.plot(t, X[i], color=PURPLE, alpha=0.2, linewidth=0.6)
ax1.plot(t, expected_value(t, x0, mu, theta), 'w--', linewidth=2.5, label=r'$E[X_t]$')
ax1.axhline(mu, color=CORAL, linewidth=2, label=f'μ = {mu}')
ax1.set_xlabel('Time', fontsize=12)
ax1.set_ylabel('X(t)', fontsize=12)
ax1.set_title('Paths Converging to Equilibrium', fontsize=13)
ax1.legend()
ax1.grid(alpha=0.3)

# Right: histogram of final values
ax2 = axes[1]
final_values = X[:, -1]

ax2.hist(final_values, bins=40, density=True, orientation='horizontal',
         alpha=0.7, color=TEAL, edgecolor='white', linewidth=0.5)

# Theoretical distribution
y_grid = np.linspace(final_values.min() - 0.5, final_values.max() + 0.5, 200)
pdf = (1 / (props.stationary_std * np.sqrt(2*np.pi))) * \
      np.exp(-0.5 * ((y_grid - mu) / props.stationary_std)**2)
ax2.plot(pdf, y_grid, 'w-', linewidth=2.5, label=r'$N(\mu, \sigma^2/2\theta)$')

ax2.axhline(mu, color=CORAL, linewidth=2)
ax2.set_xlabel('Density', fontsize=12)
ax2.set_title('Stationary\nDistribution', fontsize=13)
ax2.legend(fontsize=10)
ax2.set_ylim(ax1.get_ylim())

fig.suptitle(f'Convergence to Stationary Distribution: $X_\infty \sim N({mu}, {props.stationary_variance:.2f})$',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Part 7: 2D OU Process

In 2D, we can have **different reversion speeds per dimension**:

$$dX_t = A(\mu - X_t)dt + \sigma dW_t$$

where $A = \begin{pmatrix} \theta_x & 0 \\ 0 & \theta_y \end{pmatrix}$

In [ ]:
def simulate_ou_2d(theta_x, theta_y, mu, sigma, x0, T=10, dt=0.02, n_paths=1, seed=None):
    """Simulate 2D OU process with diagonal mean-reversion matrix."""
    if seed is not None:
        np.random.seed(seed)
    
    A = np.diag([theta_x, theta_y])
    n_steps = int(T / dt) + 1
    t = np.linspace(0, T, n_steps)
    X = np.zeros((n_paths, n_steps, 2))
    X[:, 0, :] = x0
    
    sqrt_dt = np.sqrt(dt)
    
    for i in range(1, n_steps):
        dW = np.random.randn(n_paths, 2) * sqrt_dt
        diff = mu - X[:, i-1, :]
        drift = diff @ A.T * dt
        X[:, i, :] = X[:, i-1, :] + drift + sigma * dW
    
    return t, X

print(' 2D simulation ready!')

In [ ]:
#  MODIFY THESE VALUES
theta_x = 0.3  # Reversion speed in X
theta_y = 0.3  # Reversion speed in Y

# Fixed
mu = np.array([0, 0])
sigma = 0.5
x0 = np.array([6, 5])
T = 35

# Simulate
t, X = simulate_ou_2d(theta_x, theta_y, mu, sigma, x0, T=T, n_paths=20, seed=42)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Color by time
cmap = plt.cm.plasma
n_steps = X.shape[1]

for p in range(20):
    for i in range(n_steps - 1):
        ax.plot(X[p, i:i+2, 0], X[p, i:i+2, 1], 
                color=cmap(i / n_steps), alpha=0.5, linewidth=1)

# Markers
ax.scatter(x0[0], x0[1], c=TEAL, s=300, marker='o', edgecolors='white', 
           linewidths=2, zorder=10, label=f'Start {tuple(x0)}')
ax.scatter(mu[0], mu[1], c=CORAL, s=400, marker='*', edgecolors='white',
           linewidths=2, zorder=10, label=f'μ {tuple(mu)}')

# Stationary ellipse
stat_std_x = sigma / np.sqrt(2 * theta_x)
stat_std_y = sigma / np.sqrt(2 * theta_y)
theta_grid = np.linspace(0, 2*np.pi, 100)
ax.plot(mu[0] + 2*stat_std_x*np.cos(theta_grid), 
        mu[1] + 2*stat_std_y*np.sin(theta_grid),
        'w--', linewidth=2, alpha=0.7, label='±2σ contour')

ax.set_xlabel('X₁', fontsize=13)
ax.set_ylabel('X₂', fontsize=13)
ax.set_title(f'2D OU Process\nθ_x = {theta_x}, θ_y = {theta_y}', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.set_aspect('equal')
ax.grid(alpha=0.3)

# Colorbar
sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, T))
plt.colorbar(sm, ax=ax, shrink=0.6, label='Time')
plt.show()

---

## Part 8: Anisotropic Behavior

When $\theta_x \neq \theta_y$, the process has **directional preferences**.

In [ ]:
configs = [
    (0.3, 0.3, 'Isotropic'),
    (0.8, 0.2, 'Faster in X'),
    (0.2, 0.8, 'Faster in Y'),
    (1.5, 0.1, 'Very anisotropic'),
]

mu = np.array([0, 0])
sigma = 0.5
x0 = np.array([5, 5])
T = 30

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
cmap = plt.cm.plasma

for ax, (theta_x, theta_y, title) in zip(axes.flatten(), configs):
    t, X = simulate_ou_2d(theta_x, theta_y, mu, sigma, x0, T=T, n_paths=15, seed=42)
    n_steps = X.shape[1]
    
    # Plot trajectories
    for p in range(15):
        for i in range(n_steps - 1):
            ax.plot(X[p, i:i+2, 0], X[p, i:i+2, 1],
                    color=cmap(i / n_steps), alpha=0.4, linewidth=0.8)
    
    # Markers
    ax.scatter(x0[0], x0[1], c=TEAL, s=150, marker='o', edgecolors='white', linewidths=2, zorder=10)
    ax.scatter(mu[0], mu[1], c=CORAL, s=200, marker='*', edgecolors='white', linewidths=2, zorder=10)
    
    # Ellipse
    stat_std_x = sigma / np.sqrt(2 * theta_x)
    stat_std_y = sigma / np.sqrt(2 * theta_y)
    theta_grid = np.linspace(0, 2*np.pi, 100)
    ax.plot(mu[0] + 2*stat_std_x*np.cos(theta_grid),
            mu[1] + 2*stat_std_y*np.sin(theta_grid), 'w--', linewidth=1.5, alpha=0.7)
    
    ax.set_xlabel('X₁')
    ax.set_ylabel('X₂')
    ax.set_title(f'{title}\nθ_x={theta_x}, θ_y={theta_y}', fontsize=12)
    ax.set_xlim(-5, 8)
    ax.set_ylim(-5, 8)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)

fig.suptitle('Anisotropic Mean-Reversion', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---

##  Summary: Key Takeaways

1. **Mean reversion**: The OU process always returns to μ (unlike Brownian motion)

2. **Relaxation time** $\tau = 1/\theta$: Larger θ → faster convergence

3. **Stationary distribution**: $X_\infty \sim N(\mu, \sigma^2/2\theta)$

4. **The tradeoff**: Strong reversion (high θ) fights volatility (high σ)

5. **Anisotropy**: Different eigenvalues create directional preferences

---

##  Some interesting application in Trading

- **Vasicek model**: Interest rates modeled as OU process
- **Pairs trading**: If spread is OU, compute optimal entry/exit
- **Mean-reverting vol**: Many vol models use OU dynamics
- **Parameter estimation**: Given data, estimate θ, μ, σ

Common questions:
- *"Derive E[X_t] and Var(X_t)"*
- *"What's the stationary distribution?"*
- *"Simulate and verify your formulas"*

---

##  Playground

Use this cell to experiment!

In [ ]:
# YOUR EXPERIMENTS HERE
# Try:
# - Extreme theta values (0.01 vs 10)
# - Starting very far from mu (x0 = 100)
# - Very high volatility

